# Midas Design Guide — Steel Composite Bridge Load Rating

**Companion notebook** for the corresponding chapter of the MIDAS training
manual *Design Guide for midas Civil — AASHTO LRFD*. The guide itself is
proprietary and is **not reproduced here** — this notebook contains only
original code and AASHTO LRFD / MBE article citations.

**How this notebook is used.** This notebook is an *extension of the design
guide, meant to render it unnecessary*: it carries the same design guidance
and AASHTO background in its own words, then improves on the guide with a
live Python environment for exploratory checks and quick validation, a
direct interface to Midas Civil through its API, and customization to ODOT's
design process (PSID / PSBD standard products, ODOT materials and vehicles).
Everything the guide has you do by hand in the Midas UI — coordinate entry,
side math — is demonstrated here as runnable code a designer can follow,
rerun, and modify. Where a number must come from the Midas model itself
(analysis forces, tendon losses per stage, design result tables), there is a
`TODO(midas-api)` marker — those cells get finished against the live Civil
NX JSON API when it is back online. Hand-entered values (from a standard
drawing or a hand calc) are compared via the `check()` harness below.

**Scope of this chapter:**

1. Rating vehicle set and MBE 6A factors for steel
2. Capacities at rating limit states (usually the Chapter 2 girder)
3. Force effects per vehicle from the Midas moving-load analysis (API)
4. Design load rating — HL-93 inventory / operating (Strength I + Service II)
5. Fatigue evaluation — MBE Section 7
6. Legal and permit load ratings
7. Posting evaluation and cross-check vs Midas rating tables (API)


In [ ]:
import math
import pandas as pd

# civilpy is installed editable into this env (pip install -e .)
from civilpy.structural.midas import MidasCivil, parse_result_table, envelope
from civilpy.structural.aashto.lrfd import (
    concrete, prestressed, steel, composite, distribution, lrfr, creep_shrinkage,
)

# --- Midas Civil NX connection -------------------------------------------------
# The API is not running on this machine right now. Everything below that needs
# the live model is guarded by MIDAS_ONLINE and marked TODO(midas-api).
try:
    midas = MidasCivil()
    MIDAS_ONLINE = midas.ping()
except Exception:
    midas, MIDAS_ONLINE = None, False
print("Midas Civil NX online:", MIDAS_ONLINE)

In [ ]:
# --- Validation harness --------------------------------------------------------
# Every comparison in this notebook goes through check() so the end-of-notebook
# summary shows guide value vs civilpy value side by side.
RESULTS = []

def check(label, guide_value, civilpy_value, tol=0.01, unit=""):
    """Compare a guide-reported value against the civilpy-computed one.

    tol is relative (1% default) — the guide rounds intermediate values, so
    small drift is expected; flag anything beyond tol for investigation.
    """
    if guide_value is None or civilpy_value is None:
        status = "PENDING"
        diff = None
    else:
        diff = abs(civilpy_value - guide_value) / (abs(guide_value) or 1.0)
        status = "OK" if diff <= tol else "MISMATCH"
    RESULTS.append({"check": label, "guide": guide_value, "civilpy": civilpy_value,
                    "rel diff": diff, "unit": unit, "status": status})
    print(f"[{status}] {label}: guide={guide_value} civilpy={civilpy_value} {unit}")
    return status == "OK"

def summary():
    df = pd.DataFrame(RESULTS)
    if len(df):
        n_ok = (df.status == "OK").sum()
        print(f"{n_ok}/{len(df)} checks OK, "
              f"{(df.status == 'MISMATCH').sum()} mismatches, "
              f"{(df.status == 'PENDING').sum()} pending")
    return df

## 1. Rating setup — MBE Part 6A (steel)

Steel design-load rating checks Strength I flexure and shear plus the
Service II flange-stress limit; fatigue is evaluated separately under MBE
Section 7.

In [ ]:
# TODO(guide):
GUIDE = dict(
    phi_c=None, phi_s=None,
    gamma_DC=1.25, gamma_DW=1.50,
    gamma_LL_inv=1.75, gamma_LL_op=1.35,
    adtt=None,
)
GUIDE

## 2. Capacities at the rating limit states

Reuse the Chapter 2 flexure/shear capacities (positive and negative regions)
and the Service II stress limits.

In [ ]:
# TODO(guide): phi*Mn (pos/neg), phi*Vn, and 0.95*Rh*Fyf Service II limits
# at each rated section.
pass

## 3. Force effects per rating vehicle

In [ ]:
if MIDAS_ONLINE:
    # Verified vs live Civil NX 2026-07-27: /post/TABLE selects by
    # TABLE_TYPE ("BEAMFORCE"); TABLE_NAME is just a label.
    try:
        resp = midas.result_table("Moving load envelopes",
                                  table_type="BEAMFORCE")
        display(pd.DataFrame(parse_result_table(resp)).head())
    except Exception as err:
        # a fresh/unanalyzed session (or a DB edit, or a pre-mode view
        # switch) clears results — analyze and rerun this cell
        print("no results in the session:", str(err)[-80:])
else:
    print("Midas offline — skipping Moving load envelopes (BEAMFORCE)")

## 4. Design load rating — HL-93

Strength I RFs plus the Service II stress-based rating (γLL = 1.30 both
inventory and operating for Service II per MBE 6A.4.2.1 — confirm the guide's
table).

In [ ]:
# TODO(guide): lrfr.rating_factor(...) per limit state/section;
# check() against the guide's rating summary.
pass

## 5. Fatigue evaluation — MBE Section 7

Infinite-life check first; finite remaining life if it fails. Detail
categories from the Chapter 2 design.

In [ ]:
# TODO(guide): steel.fatigue_resistance(...) with the MBE evaluation
# factors; TODO(midas-api): fatigue-truck stress ranges from the model.
pass

## 6. Legal and permit load ratings

In [ ]:
# TODO(guide): lrfr.legal_load_factor(adtt=...), lrfr.permit_load_factor(...)
# RF per vehicle; note steel typically has no stress-based legal check
# beyond Service II — confirm against the guide.
pass

## 7. Posting and Midas rating-table cross-check

In [ ]:
# TODO(midas-api): "Load Rating Result" is a PSC *design-module* table.
# Verified 2026-07-27: the design result tables (fps, c, Mcr, Av,req,
# FDL/AFDL columns) are NOT exposed through the known /post/TABLE surface —
# they need the PSC Design run configured in the Civil NX UI (design code,
# PSC design parameters, Section Manager rebar) and/or the official JSON
# manual's design TABLE_TYPE names. Analysis-side tables ARE verified:
# BEAMFORCE, BEAMSTRESSPSC (the ten-check-point stress table with
# Sig-Is(shear), Sig-Is(shear+torsion), Sig-Ps(Max/Min) columns), REACTIONG.
print("PENDING: Load Rating Result — needs the PSC Design module (see comment)")

## Validation summary

Every `check()` recorded above, in one table. `PENDING` rows are waiting on
either guide values (hand entry) or the Midas API coming back online.

In [ ]:
summary()